# Feature Extraction from Text

Machine-learning models work with numbers, not raw text. **Feature extraction** converts text into numerical values that a model can use.

In this notebook, we will learn about:

- Bag of Words;
- stop words;
- n-grams;
- TF-IDF; and
- finding similar documents with cosine similarity.

## Imports and sample text

We will use pandas to display the numerical features as tables and scikit-learn to create the features.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
texts = [
    "the red dog",
    "cat eats dog",
    "dog eats food",
    "red cat eats",
    "the hot dog",
]

## 1. Bag of Words

Bag of Words represents each text by counting its words.

- Each **row** represents one text.
- Each **column** represents one word.
- Each value tells us how many times that word appears.

The word order is not stored. This is why it is called a *bag* of words.

In [ ]:
vectorizer = CountVectorizer()

# Learn the vocabulary from the texts.
vectorizer.fit(texts)

# Convert every text into word counts.
x = vectorizer.transform(texts)

columns = vectorizer.get_feature_names_out()
pd.DataFrame(x.toarray(), columns=columns, index=texts)

,cat,dog,eats,food,hot,red,the
the red dog,0,1,0,0,0,1,1
cat eats dog,1,1,1,0,0,0,0
dog eats food,0,1,1,1,0,0,0
red cat eats,1,0,1,0,0,1,0
the hot dog,0,1,0,0,1,0,1


For example, the row for `cat eats dog` contains `1` under `cat`, `eats`, and `dog`. All other values in that row are `0`.

## `fit()` and `transform()`

These two steps have different jobs:

- `fit(texts)` learns the vocabulary.
- `transform(texts)` converts text into numbers using that vocabulary.

We can also do both steps together with `fit_transform(texts)`. Keeping them separate makes each step easier to see.

## 2. Stop words

Stop words are common words that may not help identify a text's topic. English examples include `the`, `is`, and `and`.

`stop_words="english"` tells `CountVectorizer` to remove its built-in list of common English words.

> Stop words are task-dependent. A common word may still be important in some applications.

In [ ]:
vectorizer = CountVectorizer(stop_words="english")
vectorizer.fit(texts)
x = vectorizer.transform(texts)

columns = vectorizer.get_feature_names_out()
pd.DataFrame(x.toarray(), columns=columns, index=texts)

,cat,dog,eats,food,hot,red
the red dog,0,1,0,0,0,1
cat eats dog,1,1,1,0,0,0
dog eats food,0,1,1,1,0,0
red cat eats,1,0,1,0,0,1
the hot dog,0,1,0,0,1,0


Notice that `the` is no longer a feature because it is an English stop word.

## 3. N-grams

An n-gram is a sequence of `n` neighboring words.

- A **unigram** contains one word: `hot`.
- A **bigram** contains two words: `hot dog`.

`ngram_range=(1, 2)` creates both unigrams and bigrams. This keeps some word-order information, but it also creates more features.

In [ ]:
vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(2, 3),
)

vectorizer.fit(texts)
x = vectorizer.transform(texts)

columns = vectorizer.get_feature_names_out()
pd.DataFrame(x.toarray(), columns=columns, index=texts)

,cat eats,cat eats dog,dog eats,dog eats food,eats dog,eats food,hot dog,red cat,red cat eats,red dog
the red dog,0,0,0,0,0,0,0,0,0,1
cat eats dog,1,1,0,0,1,0,0,0,0,0
dog eats food,0,0,1,1,0,1,0,0,0,0
red cat eats,1,0,0,0,0,0,0,1,1,0
the hot dog,0,0,0,0,0,0,1,0,0,0


The table now contains single words such as `hot` and two-word phrases such as `hot dog`.

## 4. TF-IDF

Bag of Words uses raw counts. TF-IDF gives each word a weight instead.

A word receives:

- a higher weight when it is important in one text; and
- a lower weight when it appears in many texts.

In simple terms:

$$TF	ext{-}IDF = 	ext{word frequency in one text} \times 	ext{word rarity across all texts}$$

The values are decimals because they are weights, not word counts.

![image.png](attachment:image.png)

![image.png](attachment:image.png)

In [ ]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1 , 2),
)

vectorizer.fit(texts)
x = vectorizer.transform(texts)

columns = vectorizer.get_feature_names_out()
tfidf_table = pd.DataFrame(x.toarray(), columns=columns, index=texts)
tfidf_table.round(2)

,cat,cat eats,dog,dog eats,eats,eats dog,eats food,food,hot,hot dog,red,red cat,red dog
the red dog,0.00,0.00,0.40,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.58,0.00,0.71
cat eats dog,0.46,0.46,0.32,0.00,0.38,0.57,0.00,0.00,0.00,0.00,0.00,0.00,0.00
dog eats food,0.00,0.00,0.29,0.52,0.35,0.00,0.52,0.52,0.00,0.00,0.00,0.00,0.00
red cat eats,0.44,0.44,0.00,0.00,0.36,0.00,0.00,0.00,0.00,0.00,0.44,0.54,0.00
the hot dog,0.00,0.00,0.37,0.00,0.00,0.00,0.00,0.00,0.66,0.66,0.00,0.00,0.00


## Bag of Words vs. TF-IDF

| Bag of Words | TF-IDF |
| --- | --- |
| Stores word counts | Stores word weights |
| Treats every word count equally | Reduces the weight of very common words |
| Simple to understand | Often more useful for comparing documents |

## 5. Finding similar documents

We can combine feature extraction with **cosine similarity** to find documents related to a query.

Cosine similarity compares two feature vectors:

- a score near `1` means very similar;
- a score near `0` means not similar.

We will use a small local dataset so the example is easy to follow and does not download anything.

In [ ]:
documents = [
    "The baseball team won the final game",
    "The player hit a baseball home run",
    "The new electric car has a strong battery",
    "Drivers should check their car tires",
    "Python is a popular programming language",
    "The programmer writes Python code",
]

query = "A baseball player won the game"

### Step 1: Convert the documents into features

We fit the vectorizer on the documents. We then use the same learned vocabulary to transform both the documents and the query.

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(documents)

document_vectors = vectorizer.transform(documents)
query_vector = vectorizer.transform([query])

The query is placed inside a list because the vectorizer expects a collection of texts, even when we have only one query.

### Step 2: Calculate similarity scores

We compare the query vector with every document vector.

In [ ]:
similarities = cosine_similarity(query_vector, document_vectors).flatten()

for document, score in zip(documents, similarities):
    print(round(score, 3), "->", document)

0.645 -> The baseball team won the final game
0.404 -> The player hit a baseball home run
0.0 -> The new electric car has a strong battery
0.0 -> Drivers should check their car tires
0.0 -> Python is a popular programming language
0.0 -> The programmer writes Python code


### Step 3: Show the most similar documents

`np.argsort()` returns the positions of the scores from smallest to largest. `[::-1]` reverses that order so the highest scores come first.

In [ ]:
sorted_indices = np.argsort(similarities)[::-1]

print("Query:", query)
print("Most similar documents:")

for index in sorted_indices[:2]:
    print(round(similarities[index], 3), "->", documents[index])

Query: A baseball player won the game
Most similar documents:
0.645 -> The baseball team won the final game
0.404 -> The player hit a baseball home run


The baseball documents should appear first because they share important words with the query. This simple method compares vocabulary; it does not fully understand meaning.